# NISAR GCOV — Covariance Dataset Analysis


# 04 — L2 GCOV Product Explorer

Now we look at the covariance terms specifically. Diagonal terms (HHHH, HVHV, VVVV, VHVH) are real-valued power; off-diagonal terms are complex and carry phase/coherence information. That split matters for everything downstream.

In [ ]:
from pathlib import Path
from nisar_utils.bootstrap import setup_workshop
WORKSHOP_ROOT = setup_workshop()

from nisar_utils.config import load_config
from nisar_utils.workflow import (
    build_profile, resolve_frequency, resolve_terms,
    resolve_aoi, resolve_window
)

cfg = load_config()
NISAR_FILE = Path(cfg["nisar_file"])
profile = build_profile(cfg)
freq = resolve_frequency(cfg, profile)
terms, diagonal_terms, off_diagonal_terms = resolve_terms(profile, freq)

print("File:", NISAR_FILE)
print("SAR family:", profile.sar_family)
print("Band:", profile.band)
print("Level:", profile.product_level)
print("Product:", profile.product_type)
print("GCOV root:", profile.gcov_root)
print("Frequency:", freq)
print("Polarization:", profile.polarization_channels)


## Geographic context

Still anchored to the same footprint from Module 01.

In [ ]:
from nisar_utils.gcov import open_gcov,get_grid_coordinates
from nisar_utils.mapping import scene_extent_wgs84,plot_scene_overview,folium_scene_map
grid=f"{profile.gcov_root}/grids/{freq}"
with open_gcov(NISAR_FILE) as f: _x,_y=get_grid_coordinates(f,grid)
scene_bounds,_=scene_extent_wgs84(_x,_y,profile.epsg)
print("Scene WGS84 extent:",scene_bounds)


In [ ]:
plot_scene_overview(_x,_y,profile.epsg,title=f"NISAR {profile.sar_family} {freq} — Geographic Context")


In [ ]:
m=folium_scene_map(_x,_y,profile.epsg,title="NISAR scene — geographic context")
m


In [ ]:
from nisar_utils.hdf5 import inspect_dataset
from nisar_utils.gcov import open_gcov

with open_gcov(NISAR_FILE) as f:
    for this_freq in profile.frequencies:
        print("\nFREQUENCY:", this_freq)
        for term in profile.covariance_terms.get(this_freq, []):
            path=f"{profile.gcov_root}/grids/{this_freq}/{term}"
            info=inspect_dataset(f,path)
            print("Summary:",term,info["shape"],info["dtype"],
                  "chunks=",info["chunks"],"compression=",info["compression"])


In [ ]:
print("\nPolarization code:", profile.polarization_code)
print("Channels:", profile.polarization_channels)
print("\nModule 04 STATUS: PASS")
